# Part 1 (v2): Workload Forecasting with LSTM + Experiments
**Project:** ML-Based Predictive Load Balancing and Auto-Scaling in Cloud Environments

Same pipeline as v1 (NASA HTTP logs → requests per 5 minutes → LSTM vs Naive and Moving Average), plus four experiments:

1. **Window size:** how much history should the model look at? (30 min, 1 h, 2 h, 3 h)
2. **Time features:** does telling the model the hour of day and day of week help?
3. **Model type:** 1-layer LSTM vs 2-layer LSTM vs GRU
4. **Prediction horizon:** predicting 5, 15 and 30 minutes ahead (matters because a new VM takes time to start)

At the end, the final model is trained with the chosen settings and `predictions.csv`, the model, the scaler and the config are saved for CloudSim Plus and the API.

Use Runtime > Change runtime type > T4 GPU in Colab.

## 1. Setup

In [ ]:
import re, os, gzip, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)

INTERVAL = '5min'   # interval size (same as one CloudSim step)
os.makedirs('outputs', exist_ok=True)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Get the dataset
NASA Kennedy Space Center HTTP logs (July 1995), from the Internet Traffic Archive.
If the FTP download fails, download `NASA_access_log_Jul95.gz` yourself and upload it when prompted.

In [ ]:
FNAME = 'NASA_access_log_Jul95.gz'
if not os.path.exists(FNAME):
    !wget -q --timeout=60 ftp://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz

if not os.path.exists(FNAME) or os.path.getsize(FNAME) < 1_000_000:
    print('Download failed, please upload NASA_access_log_Jul95.gz')
    from google.colab import files
    up = files.upload()
    FNAME = list(up.keys())[0]

print(FNAME, round(os.path.getsize(FNAME)/1e6, 1), 'MB')

## 3. Convert logs to requests per 5 minutes

In [ ]:
ts_re = re.compile(r'\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2})')
stamps = []
opener = gzip.open if FNAME.endswith('.gz') else open
with opener(FNAME, 'rt', encoding='latin-1', errors='ignore') as f:
    for line in f:
        m = ts_re.search(line)
        if m:
            stamps.append(m.group(1))

ts = pd.to_datetime(pd.Series(stamps), format='%d/%b/%Y:%H:%M:%S', errors='coerce').dropna()
print('Total requests parsed:', len(ts))

series = ts.dt.floor(INTERVAL).value_counts().sort_index()
full_idx = pd.date_range(series.index.min(), series.index.max(), freq=INTERVAL)
series = series.reindex(full_idx, fill_value=0).astype(float)
series.name = 'requests'
print('Intervals:', len(series))
series.describe()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7))
series.plot(ax=ax[0], lw=0.6, color='steelblue')
ax[0].set_title('NASA HTTP workload: requests per 5 minutes (July 1995)')
ax[0].set_ylabel('Requests')
series.iloc[:288*3].plot(ax=ax[1], lw=1, color='darkorange')
ax[1].set_title('Zoom: first 3 days (daily cycle)')
ax[1].set_ylabel('Requests')
plt.tight_layout(); plt.savefig('outputs/fig1_workload.png', dpi=200); plt.show()

## 4. Time features (new)
Web traffic follows a daily cycle, so we give the model the **time of day** and **day of week** as extra inputs.

They are encoded with sin and cos so the model sees time as a circle: 23:55 and 00:00 end up close together, which plain numbers (1435 vs 0) would not show. These values are already between -1 and 1, so they don't need scaling.

In [ ]:
df = series.to_frame()
minutes = (df.index.hour * 60 + df.index.minute).to_numpy()
dow = df.index.dayofweek.to_numpy()
df['tod_sin'] = np.sin(2 * np.pi * minutes / 1440)
df['tod_cos'] = np.cos(2 * np.pi * minutes / 1440)
df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

FEATS_BASE = ['requests']
FEATS_TIME = ['requests', 'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos']
df.head()

## 5. Helper functions
- `prepare`: chronological 70/15/15 split, scales the request count (fitted on training data only) and builds the windows.
  Each window is the last `w` intervals, and the target is the request count `h` steps after the window ends.
  Context from the previous split is added, so **every setting is tested on exactly the same test intervals** and results are directly comparable.
- `build_model`: 1-layer LSTM, 2-layer LSTM (original model) or 2-layer GRU.
- `run`: trains one model and returns its predictions plus the Naive and Moving Average baselines for the same horizon.

In [ ]:
n = len(df)
n_train, n_val = int(n * 0.70), int(n * 0.15)
test_idx = df.index[n_train + n_val:]

def make_windows(arr, w, h):
    X, y = [], []
    for i in range(w, len(arr) - h + 1):
        X.append(arr[i - w:i])
        y.append(arr[i + h - 1, 0])      # column 0 = requests
    return np.array(X), np.array(y)

def prepare(cols, w, h):
    vals = df[cols].to_numpy(dtype=float).copy()
    scaler = MinMaxScaler().fit(vals[:n_train, :1])          # fit on training data only
    vals[:, 0] = scaler.transform(vals[:, :1]).ravel()
    tr, va, te = vals[:n_train], vals[n_train:n_train + n_val], vals[n_train + n_val:]
    ctx = w + h - 1                                          # history needed before the first target
    X_tr, y_tr = make_windows(tr, w, h)
    X_va, y_va = make_windows(np.vstack([tr[-ctx:], va]), w, h)
    X_te, y_te = make_windows(np.vstack([va[-ctx:], te]), w, h)
    return X_tr, y_tr, X_va, y_va, X_te, y_te, scaler

def build_model(kind, w, n_feat):
    if kind == 'lstm1':
        body = [layers.LSTM(64), layers.Dropout(0.2)]
    elif kind == 'lstm2':
        body = [layers.LSTM(64, return_sequences=True), layers.Dropout(0.2),
                layers.LSTM(32), layers.Dropout(0.2)]
    elif kind == 'gru':
        body = [layers.GRU(64, return_sequences=True), layers.Dropout(0.2),
                layers.GRU(32), layers.Dropout(0.2)]
    else:
        raise ValueError(kind)
    model = keras.Sequential([layers.Input(shape=(w, n_feat))] + body +
                             [layers.Dense(16, activation='relu'), layers.Dense(1)])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return model

def metrics(y, p):
    mae = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    mask = y > 0
    mape = np.mean(np.abs((y[mask] - p[mask]) / y[mask])) * 100
    return round(mae, 2), round(rmse, 2), round(mape, 2)

def run(cols, w, h, kind='lstm2', epochs=60, verbose=0):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    X_tr, y_tr, X_va, y_va, X_te, y_te, scaler = prepare(cols, w, h)
    model = build_model(kind, w, len(cols))
    cb = [keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
          keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5, monitor='val_loss')]
    t0 = time.time()
    hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                     epochs=epochs, batch_size=64, callbacks=cb, verbose=verbose)
    train_sec = time.time() - t0
    inv = lambda a: scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()
    preds = {
        'Model':     np.clip(inv(model.predict(X_te, verbose=0)), 0, None),
        'Naive':     inv(X_te[:, -1, 0]),                # last known value
        'MovingAvg': inv(X_te[:, :, 0].mean(axis=1)),    # mean of the window
    }
    return dict(model=model, hist=hist, scaler=scaler, actual=inv(y_te), preds=preds, X_te=X_te, inv=inv,
                train_sec=round(train_sec, 1), epochs=len(hist.history['loss']))

def summarize(res, **info):
    row = dict(info)
    for name, p in res['preds'].items():
        mae, rmse, mape = metrics(res['actual'], p)
        row[f'{name} MAE'], row[f'{name} RMSE'], row[f'{name} MAPE'] = mae, rmse, mape
    return row

MODEL_COLS = ['Model MAE', 'Model RMSE', 'Model MAPE']

## 6. Experiment 1: Window size
How much past data should the model look at? We try 6, 12, 24 and 36 intervals (30 min to 3 hours), predicting 5 minutes ahead with request counts only.

In [ ]:
rows = []
for w in [6, 12, 24, 36]:
    print(f'Training with window = {w} ({w*5} min) ...')
    res = run(FEATS_BASE, w, 1)
    rows.append(summarize(res, window=w, history_min=w * 5, epochs=res['epochs']))

exp1 = pd.DataFrame(rows)
exp1.to_csv('outputs/exp1_window_size.csv', index=False)
BEST_W = int(exp1.loc[exp1['Model MAE'].idxmin(), 'window'])
print('Best window:', BEST_W)
display(exp1[['window', 'history_min', 'epochs'] + MODEL_COLS])

ax = exp1.plot(x='history_min', y=['Model MAE', 'Model RMSE'], kind='bar', rot=0, figsize=(8, 4),
               color=['#4c72b0', '#dd8452'])
ax.set_xlabel('History given to the model (minutes)'); ax.set_ylabel('Error (requests / 5 min)')
ax.set_title('Experiment 1: effect of window size (lower is better)'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/fig_exp1_window.png', dpi=200); plt.show()

## 7. Experiment 2: Time features
Same model and the best window from Experiment 1, with and without the time-of-day and day-of-week inputs.

In [ ]:
rows = []
for label, cols in [('Requests only', FEATS_BASE), ('Requests + time features', FEATS_TIME)]:
    print(f'Training: {label} ...')
    res = run(cols, BEST_W, 1)
    rows.append(summarize(res, features=label, n_inputs=len(cols)))

exp2 = pd.DataFrame(rows)
exp2.to_csv('outputs/exp2_time_features.csv', index=False)
USE_TIME = exp2.loc[1, 'Model MAE'] < exp2.loc[0, 'Model MAE']
BEST_COLS = FEATS_TIME if USE_TIME else FEATS_BASE
print('Time features help:', USE_TIME)
display(exp2[['features', 'n_inputs'] + MODEL_COLS])

ax = exp2.plot(x='features', y=['Model MAE', 'Model RMSE'], kind='bar', rot=0, figsize=(8, 4),
               color=['#4c72b0', '#dd8452'])
ax.set_xlabel(''); ax.set_ylabel('Error (requests / 5 min)')
ax.set_title('Experiment 2: effect of time features (lower is better)'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/fig_exp2_time_features.png', dpi=200); plt.show()

## 8. Experiment 3: Model type
1-layer LSTM vs the original 2-layer LSTM vs a 2-layer GRU (GRU has fewer gates, so it is smaller and usually faster).
Uses the best settings from Experiments 1 and 2.

In [ ]:
rows = []
for kind, label in [('lstm1', '1-layer LSTM'), ('lstm2', '2-layer LSTM'), ('gru', '2-layer GRU')]:
    print(f'Training: {label} ...')
    res = run(BEST_COLS, BEST_W, 1, kind=kind)
    rows.append(summarize(res, model=label, kind=kind, params=res['model'].count_params(),
                          train_sec=res['train_sec']))

exp3 = pd.DataFrame(rows)
exp3.to_csv('outputs/exp3_model_type.csv', index=False)
BEST_KIND = exp3.loc[exp3['Model MAE'].idxmin(), 'kind']
print('Best model type:', BEST_KIND)
display(exp3[['model', 'params', 'train_sec'] + MODEL_COLS])

ax = exp3.plot(x='model', y=['Model MAE', 'Model RMSE'], kind='bar', rot=0, figsize=(8, 4),
               color=['#4c72b0', '#dd8452'])
ax.set_xlabel(''); ax.set_ylabel('Error (requests / 5 min)')
ax.set_title('Experiment 3: model type (lower is better)'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/fig_exp3_model_type.png', dpi=200); plt.show()

## 9. Experiment 4: Prediction horizon
A new VM takes a few minutes to start, so a useful forecast has to look further ahead than that.
We predict 1, 3 and 6 steps ahead (5, 15 and 30 minutes) and compare against the baselines at each horizon.
The Naive baseline here means "the load stays the same as the last value we saw".

In [ ]:
rows = []
for h in [1, 3, 6]:
    print(f'Training: predicting {h*5} min ahead ...')
    res = run(BEST_COLS, BEST_W, h, kind=BEST_KIND)
    rows.append(summarize(res, horizon_steps=h, horizon_min=h * 5))

exp4 = pd.DataFrame(rows)
exp4['Model vs Naive (MAE % better)'] = ((exp4['Naive MAE'] - exp4['Model MAE']) / exp4['Naive MAE'] * 100).round(1)
exp4.to_csv('outputs/exp4_horizon.csv', index=False)
display(exp4[['horizon_min', 'Model MAE', 'Naive MAE', 'MovingAvg MAE',
              'Model RMSE', 'Naive RMSE', 'MovingAvg RMSE', 'Model vs Naive (MAE % better)']])

plt.figure(figsize=(8, 4))
for name, style in [('Model', 'o-'), ('Naive', 's--'), ('MovingAvg', '^--')]:
    plt.plot(exp4['horizon_min'], exp4[f'{name} MAE'], style, label=name if name != 'Model' else 'LSTM/GRU')
plt.xticks(exp4['horizon_min']); plt.xlabel('Prediction horizon (minutes ahead)')
plt.ylabel('MAE (requests / 5 min)'); plt.title('Experiment 4: error vs prediction horizon')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('outputs/fig_exp4_horizon.png', dpi=200); plt.show()

## 10. Final model
Window and features come from Experiments 1 and 2.

**Model type:** in our run the 2-layer GRU (MAE 39.27) and the 1-layer LSTM (MAE 39.62) were within 1% of each other, which is about the size of run-to-run variation. We keep the **1-layer LSTM** as the proposed model so the whole project (simulation, API, report) uses the same LSTM approach.

**Horizon:** 1 step (5 minutes). The VM boot time in CloudSim Plus is 120 s, which is shorter than one step, so a 5-minute-ahead forecast is enough to start VMs before the load arrives.

**Baselines:** the Moving Average baseline in the final table and in `predictions.csv` always uses the last **1 hour**. Its error grows with longer windows (Experiment 1), so a fixed 1-hour average keeps the comparison fair.

In [ ]:
FINAL_WINDOW   = BEST_W
FINAL_FEATURES = BEST_COLS
FINAL_KIND     = 'lstm1'  # 1-layer LSTM (see note above)
FINAL_HORIZON  = 1        # steps ahead (1 step = 5 min)
MA_STEPS       = 12       # Moving Average baseline = last 1 hour

print(f'Final: window={FINAL_WINDOW}, features={FINAL_FEATURES}, model={FINAL_KIND}, horizon={FINAL_HORIZON}')
final = run(FINAL_FEATURES, FINAL_WINDOW, FINAL_HORIZON, kind=FINAL_KIND, verbose=1)
model = final['model']
model.summary()

In [ ]:
hist = final['hist']
plt.figure(figsize=(8, 4))
plt.plot(hist.history['loss'], label='Train loss')
plt.plot(hist.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)'); plt.title('Final model training curve')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('outputs/fig2_training_curve.png', dpi=200); plt.show()

actual = final['actual']
lstm_pred, naive_pred = final['preds']['Model'], final['preds']['Naive']
ma_pred = final['inv'](final['X_te'][:, -MA_STEPS:, 0].mean(axis=1))   # fixed 1-hour moving average

results = pd.DataFrame(
    [metrics(actual, naive_pred), metrics(actual, ma_pred), metrics(actual, lstm_pred)],
    index=['Naive (last value)', f'Moving Average ({MA_STEPS*5} min)', 'LSTM (proposed)'],
    columns=['MAE', 'RMSE', 'MAPE (%)'])
results.to_csv('outputs/forecast_metrics.csv')
display(results)

k = 288 * 2   # show only 2 days
plt.figure(figsize=(14, 5))
plt.plot(test_idx[:k], actual[:k], label='Actual', color='black', lw=1.2)
plt.plot(test_idx[:k], lstm_pred[:k], label='LSTM', color='crimson', lw=1)
plt.plot(test_idx[:k], ma_pred[:k], label='Moving Avg (1 h)', color='seagreen', lw=1, ls='--')
plt.title(f'Actual vs predicted workload (test set, first 2 days, {FINAL_HORIZON*5} min ahead)')
plt.ylabel('Requests / 5 min'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('outputs/fig3_actual_vs_pred.png', dpi=200); plt.show()

ax = results[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4), rot=0, color=['#4c72b0', '#dd8452'])
ax.set_title('Forecast error comparison (lower is better)'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/fig4_error_comparison.png', dpi=200); plt.show()

## 11. Save outputs for CloudSim Plus and the API
- `predictions.csv`: one row per 5-minute step of the test period (same columns as v1). Each prediction is for that row's step, made `FINAL_HORIZON` steps earlier.
- `lstm_model.keras`, `scaler.pkl`, `config.pkl`: for the API. The scaler only covers the request count. If time features are used, the API must compute them from each input row's timestamp exactly as in Section 4. `config.pkl` lists the features, window and horizon.
- `model_weights.json`: the same weights as plain JSON, used by the FastAPI server and the browser demo in `3_api` (they run the LSTM with NumPy / JavaScript, so they don't need TensorFlow).
- `exp*_*.csv` and `fig_exp*.png`: experiment tables and graphs for the report.

In [ ]:
out = pd.DataFrame({
    'step': np.arange(len(actual)),
    'timestamp': test_idx.astype(str),
    'actual_requests': actual.round().astype(int),
    'lstm_pred': lstm_pred.round().astype(int),
    'ma_pred': ma_pred.round().astype(int),
    'naive_pred': naive_pred.round().astype(int),
})
out.to_csv('outputs/predictions.csv', index=False)

model.save('outputs/lstm_model.keras')
with open('outputs/scaler.pkl', 'wb') as f:
    pickle.dump(final['scaler'], f)
with open('outputs/config.pkl', 'wb') as f:
    pickle.dump({'window': FINAL_WINDOW, 'interval': INTERVAL, 'horizon': FINAL_HORIZON,
                 'features': FINAL_FEATURES, 'model_kind': FINAL_KIND}, f)
print(out.head()); print('Rows:', len(out))

In [ ]:
import json
assert FINAL_FEATURES == ['requests'], 'The API takes request counts only: set FINAL_FEATURES = FEATS_BASE in Section 10 before exporting'
sc = final['scaler']
layer_list = [l for l in model.layers if l.get_weights()]          # skip Dropout
weights = {
    'model': f'{FINAL_KIND} window={FINAL_WINDOW} features={FINAL_FEATURES}',
    'order': [l.name for l in layer_list],
    'types': [l.__class__.__name__.lower() for l in layer_list],    # 'lstm' or 'dense'
    'layers': {l.name: [w.tolist() for w in l.get_weights()] for l in layer_list},
    'scaler': {'min': sc.min_.tolist(), 'scale': sc.scale_.tolist(),
               'data_min': sc.data_min_.tolist(), 'data_max': sc.data_max_.tolist()},
    'window': FINAL_WINDOW,
    'interval_minutes': 5,
    'n_params': int(model.count_params()),
}
with open('outputs/model_weights.json', 'w') as f:
    json.dump(weights, f)

# check: the exported weights give the same predictions as Keras
from types import SimpleNamespace
def np_predict(x):                      # x: (window,) raw request counts
    h = (np.asarray(x, float) * sc.scale_[0] + sc.min_[0]).reshape(-1, 1)
    lstm_idx = [i for i, t in enumerate(weights['types']) if t == 'lstm']
    for i, (name, t) in enumerate(zip(weights['order'], weights['types'])):
        W = [np.array(w) for w in weights['layers'][name]]
        if t == 'lstm':
            K, R, b = W; u = R.shape[0]; hs = []; ht = np.zeros(u); c = np.zeros(u)
            for xt in h:
                z = xt @ K + ht @ R + b
                ig, fg, g, og = 1/(1+np.exp(-z[:u])), 1/(1+np.exp(-z[u:2*u])), np.tanh(z[2*u:3*u]), 1/(1+np.exp(-z[3*u:]))
                c = fg * c + ig * g; ht = og * np.tanh(c); hs.append(ht)
            h = np.array(hs) if i < lstm_idx[-1] else ht
        else:
            h = h @ W[0] + W[1]
            if i < len(weights['order']) - 1: h = np.maximum(0, h)
    return max(0.0, (float(h[0]) - sc.min_[0]) / sc.scale_[0])

X_check = final['X_te'][:50, :, 0]
keras_out = final['inv'](model.predict(final['X_te'][:50], verbose=0))
numpy_out = np.array([np_predict(sc.inverse_transform(x.reshape(-1, 1)).ravel()) for x in X_check])
print('Exported', weights['order'], '| max difference Keras vs NumPy:', np.abs(keras_out - numpy_out).max())

In [ ]:
!zip -rq outputs.zip outputs
from google.colab import files
files.download('outputs.zip')